In [1]:
pip install torch torchvision torchaudio && pip install scipy

Note: you may need to restart the kernel to use updated packages.


In [33]:
import torch
import torch.nn as nn
import cv2 as cv
from torchvision import models
from PIL import Image


In [44]:
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, ConcatDataset, random_split

# MobileNet expects specific normalization values if using pre-trained weights
transform = transforms.Compose([
    transforms.Resize((128, 128)),     # Upscale 28x28 to 128x128
    transforms.ToTensor(),             # Convert to [0, 1] range
    transforms.Normalize((0.1307,), (0.3081,)) # Standard MNIST mean/std
])

In [45]:
import random
import os
import glob
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

class FontDigitDataset(Dataset):
    """
    PyTorch Dataset for generating computerized digit images from system fonts.
    Compatible with MNIST format for mixed training.
    """
    
    def __init__(self, 
                 font_dirs=['/usr/share/fonts', '/usr/local/share/fonts'],
                 img_size=28,
                 samples_per_font=10,
                 transform=None,
                 font_size_range=(20, 24)):
        """
        Args:
            font_dirs: List of directories to search for fonts
            img_size: Size of output images (default 28x28 like MNIST)
            samples_per_font: Number of times to generate each digit per font
            transform: Optional transform to apply to images
            font_size_range: Tuple of (min, max) font sizes to use
        """
        self.img_size = img_size
        self.transform = transform
        self.font_size_range = font_size_range
        self.samples_per_font = samples_per_font
        
        # Find all TrueType fonts
        self.fonts = self._find_fonts(font_dirs)
        
        if not self.fonts:
            raise RuntimeError(f"No fonts found in {font_dirs}. Please check font directories.")
        
        print(f"Found {len(self.fonts)} fonts")
        
        # Generate dataset indices: (digit, font_idx, sample_idx)
        self.samples = []
        for digit in range(10):
            for font_idx in range(len(self.fonts)):
                for sample_idx in range(samples_per_font):
                    self.samples.append((digit, font_idx, sample_idx))
    
    def _find_fonts(self, font_dirs):
        """Find all TrueType font files in specified directories"""
        fonts = []
        for font_dir in font_dirs:
            if os.path.exists(font_dir):
                # Search for .ttf and .otf files
                fonts.extend(glob.glob(os.path.join(font_dir, '**/*.ttf'), recursive=True))
                fonts.extend(glob.glob(os.path.join(font_dir, '**/*.otf'), recursive=True))
        
        # Filter out fonts that might cause issues
        valid_fonts = []
        for font_path in fonts:
            try:
                # Try to load the font to ensure it's valid
                ImageFont.truetype(font_path, 20)
                valid_fonts.append(font_path)
            except Exception:
                pass  # Skip problematic fonts
        
        return valid_fonts
    
    def _generate_digit_image(self, digit, font_path, seed):
        """Generate a single digit image with the specified font"""
        random.seed(seed)
        
        # Randomize font size slightly for variation
        font_size = random.randint(*self.font_size_range)
        
        try:
            font = ImageFont.truetype(font_path, font_size)
        except Exception:
            # Fallback to default font if loading fails
            font = ImageFont.load_default()
        
        # Create image with white background
        img = Image.new('L', (self.img_size, self.img_size), color=255)
        draw = ImageDraw.Draw(img)
        
        text = str(digit)
        
        # Get text bounding box for centering
        bbox = draw.textbbox((0, 0), text, font=font)
        text_width = bbox[2] - bbox[0]
        text_height = bbox[3] - bbox[1]
        
        # Add slight random offset for variation
        offset_x = random.randint(-2, 2)
        offset_y = random.randint(-2, 2)
        
        # Calculate position to center text
        x = (self.img_size - text_width) // 2 - bbox[0] + offset_x
        y = (self.img_size - text_height) // 2 - bbox[1] + offset_y
        
        # Draw text in black
        draw.text((x, y), text, fill=0, font=font)
        
        return img
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        digit, font_idx, sample_idx = self.samples[idx]
        font_path = self.fonts[font_idx]
        
        # Use a deterministic seed based on indices for reproducibility
        seed = hash((digit, font_idx, sample_idx)) % (2**32)
        
        img = self._generate_digit_image(digit, font_path, seed)
        
        if self.transform:
            img = self.transform(img)
        
        return img, digit

In [184]:
# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)
print(torch.cuda.is_available())

cuda
True


In [185]:
# Initialise the model
import torch.optim as optim
# 1. Load the pre-trained MobileNetV2
model = models.mobilenet_v3_small(pretrained=True)

# 2. Modify the first layer to accept grayscale (1 channel)
# Original: Conv2d(3, 32, kernel_size=(3, 3)...)
existing_layer = model.features[0][0]

# # 4. Replace the classifier "Head" 
# # MobileNetV2's classifier is a Dropout + Linear layer
# model.avgpool = nn.AdaptiveAvgPool2d((1, 1))

# Loss Function: CrossEntropy is standard for multi-class (0-9 digits)
criterion = nn.CrossEntropyLoss()


/home/jha/Documents/Projects/websites/sudoku/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/jha/Documents/Projects/websites/sudoku/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Small_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [40]:
# 3. Freeze all parameters (prevent gradients from updating base weights)
for name, param in model.named_parameters():
    # freeze everything except the classifier head
    if "classifier" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

In [6]:
# This is for fine tuning
for name, param in model.named_parameters():
        param.requires_grad = True

In [186]:
# Optimizer: Adam is a safe, fast-learning default
# We only pass parameters where requires_grad=True
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

num_ftrs = model.classifier[3].in_features
print(f"num features: {num_ftrs}")
print(f"Classifier: {model.classifier}")
model.classifier[3] = nn.Linear(num_ftrs, 10) # 10 classes for digits 0-9
print(f"New Classifier: {model.classifier}")
model.to(device)

num features: 1024
Classifier: Sequential(
  (0): Linear(in_features=576, out_features=1024, bias=True)
  (1): Hardswish()
  (2): Dropout(p=0.2, inplace=True)
  (3): Linear(in_features=1024, out_features=1000, bias=True)
)
New Classifier: Sequential(
  (0): Linear(in_features=576, out_features=1024, bias=True)
  (1): Hardswish()
  (2): Dropout(p=0.2, inplace=True)
  (3): Linear(in_features=1024, out_features=10, bias=True)
)


MobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): SqueezeExcitation(
          (avgpool): AdaptiveAvgPool2d(output_size=1)
          (fc1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
          (fc2): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
          (activation): ReLU()
          (scale_activation): Hardsigmoid()
        )
        (2): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(1, 1), 

In [161]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.005):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter = 0
        self.should_stop = False

    def step(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            return True   # improvement
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
            return False
        
# Custom Transform function
class AddGaussianNoise:
    def __init__(self, mean=0.0, std=0.05):
        self.mean = mean
        self.std = std

    def __call__(self, tensor):
        return tensor + torch.randn_like(tensor) * self.std + self.mean

In [159]:
# 1. Load the "Big" training set
# Initialise the Training dataset
train_transform = transforms.Compose([
        transforms.Resize(128),
        transforms.Grayscale(num_output_channels=3),  # Convert grayscale to 3-channel
        transforms.RandomRotation(15),  # Data augmentation
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        AddGaussianNoise(std=0.03),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.5)
    ])

full_train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=train_transform)
svhn_train = datasets.SVHN(root='./data', split='train', download=True, 
                                   transform=train_transform)
# font_dataset = FontDigitDataset(
#     font_dirs=['/usr/share/fonts', '/usr/local/share/fonts'],
#     img_size=28,
#     samples_per_font=10,  # Generate 10 variations per digit per font
#     transform=train_transform,
#     font_size_range=(20, 24)
# ) 

full_train_dataset = ConcatDataset([full_train_dataset, svhn_train])
print(f"train: {len(full_train_dataset)}")
# 2. Define the sizes (e.g., 50,000 for training, 10,000 for validation)
train_size = int(len(full_train_dataset) * 0.7) # 70%
val_size = len(full_train_dataset) - train_size 

# 3. Create the two separate dataset objects
train_subset, val_subset = random_split(full_train_dataset, [train_size, val_size])

# 4. Now create your DataLoaders
train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=64, shuffle=False)

train: 133257


In [53]:
test_transform = transforms.Compose([ 
    transforms.Resize(128), 
    transforms.Grayscale(num_output_channels=3), 
    transforms.ToTensor(), 
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
])


full_test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=test_transform)
svhn_test = datasets.SVHN(root='./data', split='test', download=True, 
                                   transform=test_transform)
# font_dataset = FontDigitDataset(
#     font_dirs=['/usr/share/fonts', '/usr/local/share/fonts'],
#     img_size=28,
#     samples_per_font=2,  # Generate 2 variations per digit per font
#     transform=test_transform,
#     font_size_range=(20, 24)
# ) 

full_test_dataset = ConcatDataset([full_test_dataset, svhn_test])
print(f"train: {len(full_test_dataset)}")
# 2. Define the sizes (e.g., 50,000 for training, 10,000 for validation)
test_size = int(len(full_test_dataset) * 0.7) # 70%
val_size = len(full_test_dataset) - test_size

# 3. Create the two separate dataset objects
test_subset, val_subset = random_split(full_test_dataset, [test_size, val_size])

# 4. Now create your DataLoaders
test_loader = DataLoader(test_subset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=64, shuffle=False)



train: 36032


In [54]:
def train_model(model, train_loader, val_loader, optimizer, epochs=5):
    early_stopping = EarlyStopping(patience=7)
    for epoch in range(epochs):
        # --- TRAINING PHASE ---
        model.train()
        train_loss = 0.0
        
        for images, labels in train_loader:
            # 1. Move data to the same device as the model
            images = images.to(device)
            labels = labels.to(device)

            # 2. Forward pass: Get predictions
            outputs = model(images)
            
            # 3. Calculate Loss
            loss = criterion(outputs, labels)
            # 4. Backward pass: Calculate gradients and update
            optimizer.zero_grad() # Clear previous gradients
            loss.backward()       # Compute new gradients
            optimizer.step()      # Update weights
            train_loss += loss.item()

        # --- VALIDATION PHASE ---
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad(): # Disable gradient calculation for speed/memory
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                _, predicted = torch.max(outputs, 1)
                correct += (predicted == labels).sum().item()
                total += labels.size(0)

        train_loss /= len(train_loader)
        val_loss /= len(val_loader)

        print(f"Epoch {epoch}: train={train_loss:.4f}, val={val_loss:.4f} accuracy: {float(correct / total) * 100:.4f}")

        improved = early_stopping.step(val_loss)
        if improved:
            torch.save(model.state_dict(), "best_model.pt")

        if early_stopping.should_stop:
            print("Early stopping triggered")
            break



In [141]:
train_model(model, train_loader, val_loader, optimizer, epochs=200)

KeyboardInterrupt: 

In [187]:
# Load the best model from training
model.load_state_dict(torch.load("best_model.pt"))

<All keys matched successfully>

In [188]:
def test_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f'Final Test Accuracy: {100 * correct / total:.2f}%')

test_model(model, test_loader)

Final Test Accuracy: 97.08%


In [189]:
# Example: Fine-tuning everything (Stage 2)

for param in model.parameters():
    param.requires_grad = True

# Use a 10x or 100x smaller learning rate for the base
optimizer = torch.optim.Adam(model.parameters(), lr=0.00001)


In [190]:
train_model(model, train_loader, val_loader, optimizer, epochs=200)

Epoch 0: train=0.1605, val=0.1710 accuracy: 94.7746
Epoch 1: train=0.1614, val=0.1695 accuracy: 94.7021
Epoch 2: train=0.1604, val=0.1629 accuracy: 94.9097
Epoch 3: train=0.1594, val=0.1678 accuracy: 94.8071
Epoch 4: train=0.1584, val=0.1662 accuracy: 94.9347
Epoch 5: train=0.1545, val=0.1689 accuracy: 94.8096
Epoch 6: train=0.1548, val=0.1681 accuracy: 94.8397
Epoch 7: train=0.1531, val=0.1644 accuracy: 94.9172
Epoch 8: train=0.1523, val=0.1681 accuracy: 94.8647
Epoch 9: train=0.1537, val=0.1669 accuracy: 94.7796
Early stopping triggered


In [191]:
test_model(model, test_loader)

Final Test Accuracy: 97.22%


In [192]:
# Define the file path (use .pth or .pt extension)
PATH = "best_model.pt"

# Save the weights
torch.save(model.state_dict(), PATH)
print("Model weights saved!")

Model weights saved!


In [239]:
import time

image_loc = "../../images/"
number_loc = image_loc + "9.png"

In [245]:

# Set model to evaluation mode
model.eval()
device = torch.device("cpu")
# Load model weights onto the same device as the model
model.load_state_dict(torch.load(PATH, map_location=device))
torch.set_num_threads(1)

model.to(device)

old_time = time.time()
# transform the image to 128 x 128
# Load the image using OpenCV (cv2)
image = cv.imread(number_loc)  # Replace with your image path

# Convert BGR to RGB (OpenCV loads images in BGR format)
image = cv.cvtColor(image, cv.COLOR_BGR2RGB)

transform = transforms.Compose([
    transforms.ToPILImage(),  # Convert from OpenCV (NumPy array) to PIL image
    transforms.Resize(128),     # Upscale 28x28 to 128x128
    transforms.ToTensor(),             # Convert to [0, 1] range
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # match training normalization
])

resized_image = transform(image)
image_tensor = resized_image.unsqueeze(0).to(device)  # Shape will be (1, 3, 128, 128) and moved to device
print(image_tensor.shape)

# Perform the prediction
with torch.no_grad():  # We don't need gradients for inference
    output = model(image_tensor)

print(f"Output: {output}")
# Get the predicted class
_, predicted_class = torch.max(output, 1)  # Get the index of the maximum score

time_diff = time.time() - old_time

# If you have class labels, you can map the index to the class name
# For example, if you have a list of class names:

class_names = ['0','1', '2', '3', '4', '5','6', '7', '8', '9']  # Replace with your actual class names
predicted_class_name = class_names[predicted_class.item()]

print(f'Image: {number_loc}, Predicted class: {predicted_class_name} in {time_diff} secs')


torch.Size([1, 3, 138, 128])
Output: tensor([[-0.1028, -1.8840, -1.3705, -1.5929, -1.7670, -0.6625, -4.6389, -3.9531,
         -0.2270,  8.8649]])
Image: ../../images/9.png, Predicted class: 9 in 0.009956836700439453 secs
